In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, train_size = 0.8, random_state=42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()
y_train, x_train = train_df[:, 0], train_df[:, 1:].astype(np.float32)
y_test, x_test = test_df[:, 0], test_df[:, 1:].astype(np.float32)

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))

In [ ]:
x_train.shape

In [ ]:
eps = 1e-6
def relu(x):
    return x.clamp(min=0)

def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0)
    x_norm = (x - batch_mean)/torch.sqrt(batch_var + eps)

In [ ]:
def softmax(x):
    z = torch.exp(x)
    y = z/torch.sum(z, dim=1)
    return y

In [ ]:
class MLP:
    def __init__(self):
        self.W1 = (torch.rand(256, 784) - 0.5)/torch.sqrt(torch.tensor(784))
        self.b1 = (torch.rand(256) - 0.5)/torch.sqrt(torch.tensor(784))
        self.W2 = (torch.rand(100, 256) - 0.5)/torch.sqrt(torch.tensor(256))
        self.b2 = (torch.rand(100) - 0.5)/torch.sqrt(torch.tensor(256))
        self.W3 = (torch.rand(10, 100) - 0.5)/torch.sqrt(torch.tensor(100))
        self.b3 = (torch.rand(10) - 0.5)/torch.sqrt(torch.tensor(100))
        self.W1.to(device).requires_grad_()
        self.b1.to(device).requires_grad_()
        self.W2.to(device).requires_grad_()
        self.b2.to(device).requires_grad_()
        self.W3.to(device).requires_grad_()
        self.b3.to(device).requires_grad_()

    def __call__(self, x):
        y1 = relu(x@self.W1.t() + self.b1)
        y2 = relu(y1@self.W2.t() + self.b2)
        logits = y2@self.W3.t() + self.b3
        return logits

In [ ]:
def cross_entropy(logits, yb):
    logprobs = torch.log_softmax(logits, dim=1)
    loss = -logprobs[torch.arange(len(yb)), yb].mean()
    return loss

In [ ]:
model = MLP()

In [ ]:
model(x_train[0:1])

In [ ]:
class TrainDataset(Dataset):
    def __init__(self, x, y):
        self.x = x/255.0
        self.y = y
        

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

class TestDataset(Dataset):
    def __init__(self, x, y):
        self.x = x/255.0
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

In [ ]:
train_ds = TrainDataset(x_train, y_train)
test_ds = TestDataset(x_test, y_test)
train_dl = DataLoader(train_ds, batch_size=64)
test_dl = DataLoader(test_ds, batch_size=64)

In [ ]:
EPOCHS = 100
lr= 5e-3
model = MLP()

for epoch in range(EPOCHS):
    batch_loss = 0.0
    cnt = 0
    for xb, yb in train_dl:
        cnt += 1
        ypred = model(xb)
        
        loss = cross_entropy(ypred, yb)
        batch_loss += loss.item()
        loss.backward()
        with torch.no_grad():
            model.W1 -= lr*model.W1.grad
            model.b1 -= lr*model.b1.grad
            model.W2 -= lr*model.W2.grad
            model.b2 -= lr*model.b2.grad
            model.W3 -= lr*model.W3.grad
            model.b3 -= lr*model.b3.grad
            model.W1.grad.zero_()
            model.b1.grad.zero_()
            model.W2.grad.zero_()
            model.b2.grad.zero_()
            model.W3.grad.zero_()
            model.b3.grad.zero_()
    print(batch_loss/cnt)

# Accuracy

In [ ]:
(torch.argmax(model(x_test), -1) == y_test).to(float).mean()